In [0]:
from scripts.silver import silver_loader
from scripts.utils import silver_utils

print(dir(silver_loader))
print(dir(silver_utils))

In [0]:
from pyspark.sql.functions import (
    col, when, sum, lit, trim, lower, upper,
    to_date, current_timestamp,round
)
from datetime import datetime
import uuid

In [0]:
df=silver_loader.get_incremental(spark,'products',"id")


In [0]:
silver_utils.check_schema(df)

In [0]:
df = (
    df.drop("images")
      .drop("description")
      .drop("slug")
      .drop("category_image")
      .drop("category_slug")
)


#check changes
silver_utils.check_schema(df)

In [0]:
df=silver_utils.cast_types(df,{
    "id" : "long",
    "price" : "long",
    "creationAt" : "timestamp",
    "updatedAt" : "timestamp",
    "title" : "string",
    "category_id" : "long",
    "category_name" : "string"
})

#check changes
print(df.dtypes)

In [0]:
df = silver_utils.rename_col(df,"id","product_id")
df = silver_utils.rename_col(df,"title","product_title")

#check changes
df.columns

In [0]:
silver_utils.null_profiling(df,"products")

df= silver_utils.handle_nulls_drop(
    df,
    drop_cols=['product_id', "product_title", "category_id"]
    )

In [0]:
df = silver_utils.handle_duplicates(df,["product_id"])

In [0]:
df = silver_utils.standardize_strings(
    df,{
    "product_title" : lambda c : lower(trim(c)),
    "category_name" : lambda c : lower(trim(c))
    }
)


In [0]:
checks = [
    #nulls
    (
        "product_id",
        "product_id not null",
        col("product_id").isNotNull()
    ),
    (
        "product_title",
        "product_title not null",
        col("product_title").isNotNull()
    ),
    (
        "category_id",
        "category_id not null",
        col("category_id").isNotNull()
    ),

    #business rules
    (
        "price", "must be >= 0", col("price") >= 0
    ),
    (
        "updatedAt", "must be >= creationAt", col("updatedAt") >= col("creationAt")
    ),
    (
        "ingestion_time", "must be >= updatedAt", col("ingestion_time") >= col("updatedAt")
    )
]

dq_table = silver_utils.build_dq_table(
    spark=spark,
    df=df,
    checks=checks,
    table_name="products_api",
    warn_threshold=5.0
)

dq_table.write.mode("append").saveAsTable("workspace.ecommerce_silver.products_api_dq")
dq_saved = spark.read.table("workspace.ecommerce_silver.products_api_dq")
display(dq_saved)

In [0]:
df = df.withColumn("price_with_tax", round( col("price") * 1.14,2)) #consider tax is 14%

#check changes
print(df.columns)

In [0]:
df = silver_utils.add_silver_metadata(df)

#check changes
print(df.columns)

In [0]:
# Append new rows into silver
df.write.format("delta").mode("append").saveAsTable("workspace.ecommerce_silver.products")
print(" Data written to silver table successfully")

In [0]:
%sql
--sanity check
SELECT * FROM workspace.ecommerce_silver.products LIMIT 10